# UndertriAI — GRPO Training on Bail Assessment Environment

This notebook trains **Qwen2.5-3B-Instruct** using GRPO against the live UndertriAI environment.

**Environment**: https://draken1606-undertrial-ai.hf.space  
**GitHub**: https://github.com/Faiz-1606/Undertrial  
**HF Space**: https://huggingface.co/spaces/Draken1606/undertrial-ai

## What this notebook does
1. Installs Unsloth + TRL
2. Loads episode data from the environment
3. Runs GRPO training with 5 reward components
4. Plots reward curves showing agent improvement
5. Saves the trained model adapter

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets httpx matplotlib

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────
import os, json, re, random, requests
from pathlib import Path
import torch
from datasets import Dataset
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a0d1a'
matplotlib.rcParams['axes.facecolor']   = '#111827'
matplotlib.rcParams['axes.edgecolor']   = '#1e2d45'
matplotlib.rcParams['text.color']       = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor']  = '#94a3b8'
matplotlib.rcParams['xtick.color']      = '#64748b'
matplotlib.rcParams['ytick.color']      = '#64748b'
matplotlib.rcParams['grid.color']       = '#1e2d45'

BASE_URL = 'https://draken1606-undertrial-ai.hf.space'
print('Environment URL:', BASE_URL)
r = requests.get(BASE_URL + '/health')
print('Health:', r.json())

In [ ]:
# ── Cell 3: Load episode dataset from GitHub ───────────────────────────────
import urllib.request

EPISODES_BASE = 'https://raw.githubusercontent.com/Faiz-1606/Undertrial/main/data/episodes/'

def load_jsonl_from_url(url):
    with urllib.request.urlopen(url) as f:
        return [json.loads(line) for line in f.read().decode().splitlines() if line.strip()]

episodes = []
for stage in [1, 2, 3, 4]:
    url = EPISODES_BASE + f'episodes_stage_{stage}.jsonl'
    try:
        eps = load_jsonl_from_url(url)
        episodes.extend(eps)
        print(f'Stage {stage}: {len(eps)} episodes')
    except Exception as e:
        print(f'Stage {stage}: failed ({e})')

print(f'Total: {len(episodes)} episodes loaded')
random.shuffle(episodes)

# Use first 100 for fast training demo
TRAIN_EPISODES = episodes[:100]
print(f'Training on {len(TRAIN_EPISODES)} episodes')

In [ ]:
# ── Cell 4: System prompt and case formatter ───────────────────────────────
SYSTEM_PROMPT = """You are a senior judicial clerk AI preparing a bail assessment memo for the judge.
Read the case carefully and produce a structured assessment.

Your response MUST be in this exact XML format:

<think>
[Step-by-step legal reasoning: charges, max sentence, custody time, flight risk, precedents]
</think>

<memo>
<flight_risk>Low|Medium|High</flight_risk>
<flight_risk_justification>[specific reasons from facts]</flight_risk_justification>
<statutory_eligible>true|false</statutory_eligible>
<statutory_computation>[Section X max Y years threshold Z months served W months]</statutory_computation>
<grounds_for_bail>
  <ground>[ground 1]</ground>
</grounds_for_bail>
<grounds_against_bail>
  <ground>[ground 1]</ground>
</grounds_against_bail>
<recommended_outcome>Bail Granted|Bail Denied</recommended_outcome>
<recommended_conditions>
  <condition>[condition if granted]</condition>
</recommended_conditions>
</memo>"""

def format_prompt(ep):
    ipc = ', '.join(ep.get('ipc_sections', []))
    pros = '\n'.join(f'  * {a}' for a in ep.get('prosecution_arguments', []))
    defe = '\n'.join(f'  * {a}' for a in ep.get('defence_arguments', []))
    return f"""BAIL CASE: {ep.get('case_title')}
Court: {ep.get('court')} | Date: {ep.get('date')}
Sections: {ipc} | Crime: {ep.get('crime_type')}
Custody: {ep.get('custody_months')} months | Max sentence: {ep.get('max_sentence_years')} years

CHARGE SHEET:
{ep.get('charge_sheet')}

PROSECUTION:
{pros}

DEFENCE:
{defe}"""

# Build HuggingFace dataset
def make_dataset(episodes):
    rows = []
    for ep in episodes:
        rows.append({
            'prompt': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': format_prompt(ep)},
            ],
            'episode': json.dumps(ep),
        })
    return Dataset.from_list(rows)

train_dataset = make_dataset(TRAIN_EPISODES)
print('Dataset:', train_dataset)

In [ ]:
# ── Cell 5: Reward functions ───────────────────────────────────────────────
def extract_tag(text, tag):
    m = re.search(rf'<{tag}>(.*?)</{tag}>', text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else ''

def reward_outcome(completions, prompts, **kwargs):
    """40% weight: Did the agent match the HC decision?"""
    scores = []
    for i, comp in enumerate(completions):
        text = comp[0]['content'] if isinstance(comp, list) else comp
        ep = json.loads(prompts[i].get('episode', '{}')) if isinstance(prompts[i], dict) else {}
        gt_outcome = ep.get('ground_truth', {}).get('outcome', '')
        agent_outcome = extract_tag(text, 'recommended_outcome')
        if not agent_outcome:
            scores.append(0.0); continue
        if agent_outcome.lower().strip() == gt_outcome.lower().strip():
            scores.append(1.0)
        elif ('grant' in agent_outcome.lower()) == ('grant' in gt_outcome.lower()):
            scores.append(0.8)
        else:
            scores.append(0.0)
    return scores

def reward_format(completions, **kwargs):
    """Format compliance: has <think> and <memo> blocks"""
    scores = []
    for comp in completions:
        text = comp[0]['content'] if isinstance(comp, list) else comp
        has_think = bool(re.search(r'<think>.*?</think>', text, re.DOTALL))
        has_memo  = bool(re.search(r'<memo>.*?</memo>', text, re.DOTALL))
        has_outcome = bool(re.search(r'<recommended_outcome>', text))
        has_flight  = bool(re.search(r'<flight_risk>', text))
        score = sum([has_think, has_memo, has_outcome, has_flight]) / 4.0
        scores.append(score)
    return scores

def reward_flight_risk(completions, prompts, **kwargs):
    """20% weight: flight risk assessment accuracy"""
    LOW_KW  = ['not a flight risk','local ties','permanent resident','family','no prior']
    HIGH_KW = ['abscond','organized crime','intimidat','repeat offend','nexus']
    scores = []
    for i, comp in enumerate(completions):
        text = comp[0]['content'] if isinstance(comp, list) else comp
        ep = json.loads(prompts[i].get('episode', '{}')) if isinstance(prompts[i], dict) else {}
        gt_risk = ep.get('ground_truth', {}).get('implicit_flight_risk', 'Low')
        agent_risk = extract_tag(text, 'flight_risk')
        if agent_risk.lower() == gt_risk.lower():
            scores.append(1.0)
        elif (agent_risk.lower() in ['low','medium']) == (gt_risk.lower() in ['low','medium']):
            scores.append(0.5)
        else:
            scores.append(0.0)
    return scores

def reward_bias_penalty(completions, prompts, **kwargs):
    """Anti-bias: penalize if ground truth has bias_flag=True but agent ignores parity"""
    scores = []
    BIAS_PHRASES = ['character', 'community standing', 'religion', 'caste', 'background of the accused']
    for i, comp in enumerate(completions):
        text = comp[0]['content'] if isinstance(comp, list) else comp
        ep = json.loads(prompts[i].get('episode', '{}')) if isinstance(prompts[i], dict) else {}
        bias_case = ep.get('ground_truth', {}).get('bias_flag', False)
        if bias_case:
            # Agent should mention parity or reject character grounds
            mentions_parity = 'parity' in text.lower() or 'similarly situated' in text.lower()
            uses_bias_phrase = any(p in text.lower() for p in BIAS_PHRASES)
            if mentions_parity and not uses_bias_phrase:
                scores.append(0.0)  # no penalty — agent handled bias correctly
            elif uses_bias_phrase:
                scores.append(1.0)  # full penalty
            else:
                scores.append(0.3)  # partial penalty
        else:
            scores.append(0.0)  # no bias case — no penalty
    return scores

def combined_reward(completions, prompts, **kwargs):
    """Master reward: weighted combination of all components"""
    outcome = reward_outcome(completions, prompts, **kwargs)
    fmt     = reward_format(completions, **kwargs)
    flight  = reward_flight_risk(completions, prompts, **kwargs)
    bias    = reward_bias_penalty(completions, prompts, **kwargs)
    rewards = [
        0.4*o + 0.2*f + 0.2*fr + 0.1*fm - 0.3*b
        for o, f, fr, fm, b in zip(outcome, flight, flight, fmt, bias)
    ]
    return rewards

print('Reward functions defined.')

In [ ]:
# ── Cell 6: Load model with Unsloth ───────────────────────────────────────
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct'
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded with LoRA adapters.')

In [ ]:
# ── Cell 7: Baseline evaluation BEFORE training ────────────────────────────
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

def evaluate_model(model, tokenizer, episodes, n=20):
    """Run n episodes and return mean reward + component scores."""
    sample = random.sample(episodes, min(n, len(episodes)))
    total_rewards, outcomes, flights, formats = [], [], [], []

    for ep in sample:
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': format_prompt(ep)},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors='pt'
        ).to('cuda')
        with torch.no_grad():
            out = model.generate(inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
        text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

        comp = [{'content': text}]
        prompt_dict = {'episode': json.dumps(ep)}

        r_out = reward_outcome([comp], [prompt_dict])[0]
        r_fmt = reward_format([comp])[0]
        r_flt = reward_flight_risk([comp], [prompt_dict])[0]
        r_bias = reward_bias_penalty([comp], [prompt_dict])[0]
        r_total = 0.4*r_out + 0.2*r_flt + 0.2*r_flt + 0.1*r_fmt - 0.3*r_bias

        total_rewards.append(r_total)
        outcomes.append(r_out)
        flights.append(r_flt)
        formats.append(r_fmt)

    return {
        'mean_reward': sum(total_rewards)/len(total_rewards),
        'outcome_acc': sum(outcomes)/len(outcomes),
        'flight_acc':  sum(flights)/len(flights),
        'format_acc':  sum(formats)/len(formats),
    }

print('Running baseline evaluation (before training)...')
baseline = evaluate_model(model, tokenizer, TRAIN_EPISODES, n=20)
print('BASELINE:', baseline)

In [ ]:
# ── Cell 8: GRPO Training ─────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel
FastLanguageModel.for_training(model)

training_args = GRPOConfig(
    output_dir='./undertrial_grpo_output',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,           # rollouts per prompt
    max_prompt_length=1024,
    max_completion_length=512,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=50,
    report_to='none',
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_outcome,
        reward_format,
        reward_flight_risk,
        reward_bias_penalty,
    ],
    args=training_args,
    train_dataset=train_dataset,
)

print('Starting GRPO training...')
trainer.train()
print('Training complete!')

In [ ]:
# ── Cell 9: Post-training evaluation ──────────────────────────────────────
FastLanguageModel.for_inference(model)

print('Running post-training evaluation...')
post = evaluate_model(model, tokenizer, TRAIN_EPISODES, n=20)
print('POST-TRAINING:', post)

print()
print('=== IMPROVEMENT SUMMARY ===')
print(f"Total Reward:   {baseline['mean_reward']:.3f} → {post['mean_reward']:.3f}  (+{post['mean_reward']-baseline['mean_reward']:.3f})")
print(f"Outcome Match:  {baseline['outcome_acc']:.3f} → {post['outcome_acc']:.3f}")
print(f"Flight Risk:    {baseline['flight_acc']:.3f} → {post['flight_acc']:.3f}")
print(f"Format Score:   {baseline['format_acc']:.3f} → {post['format_acc']:.3f}")

In [ ]:
# ── Cell 10: Plot reward curves ────────────────────────────────────────────
import numpy as np

# Extract training log
log = trainer.state.log_history
steps   = [e['step']   for e in log if 'reward' in e]
rewards = [e['reward'] for e in log if 'reward' in e]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0d1a')

# Left: training reward curve
ax1 = axes[0]
if steps:
    ax1.plot(steps, rewards, color='#6366f1', linewidth=2, alpha=0.8)
    # Smoothed
    if len(rewards) > 5:
        smooth = np.convolve(rewards, np.ones(5)/5, mode='valid')
        ax1.plot(steps[2:-2], smooth, color='#14b8a6', linewidth=2, label='Smoothed')
    ax1.set_xlabel('Training Step')
    ax1.set_ylabel('Reward')
    ax1.set_title('Training Reward Curve', color='#e2e8f0', pad=12)
    ax1.grid(True, alpha=0.2)
    ax1.legend(facecolor='#111827', edgecolor='#1e2d45', labelcolor='#94a3b8')
else:
    ax1.text(0.5, 0.5, 'No steps logged', ha='center', va='center', transform=ax1.transAxes)

# Right: before vs after bar chart
ax2 = axes[1]
metrics = ['Total\nReward', 'Outcome\nMatch', 'Flight\nRisk', 'Format\nScore']
before  = [baseline['mean_reward'], baseline['outcome_acc'], baseline['flight_acc'], baseline['format_acc']]
after   = [post['mean_reward'],     post['outcome_acc'],     post['flight_acc'],     post['format_acc']]

x = np.arange(len(metrics))
w = 0.35
ax2.bar(x - w/2, before, w, label='Before Training', color='#3730a3', alpha=0.8)
ax2.bar(x + w/2, after,  w, label='After Training',  color='#6366f1', alpha=0.9)
ax2.set_xticks(x)
ax2.set_xticklabels(metrics, fontsize=9)
ax2.set_ylim(0, 1.1)
ax2.set_title('Before vs After Training', color='#e2e8f0', pad=12)
ax2.legend(facecolor='#111827', edgecolor='#1e2d45', labelcolor='#94a3b8')
ax2.grid(True, alpha=0.2, axis='y')

plt.suptitle('UndertriAI — GRPO Training Results', color='#e2e8f0', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight', facecolor='#0a0d1a')
plt.show()
print('Saved: reward_curve.png')

In [ ]:
# ── Cell 11: Save model ────────────────────────────────────────────────────
model.save_pretrained('undertrial_lora_adapter')
tokenizer.save_pretrained('undertrial_lora_adapter')

# Save as merged 16-bit for inference (use Unsloth's safe merge path)
model.save_pretrained_merged(
    'undertrial_merged',
    tokenizer,
    save_method='merged_16bit',
)
print('Model saved!')

# Download reward curve
from google.colab import files
files.download('reward_curve.png')